**MGMT298D: Science and Strategy of AI**
# Week 5: Transfer Learning

#### We build a handbags-vs-shoes classifier two ways: first training a small CNN entirely from scratch, then freezing a pretrained VGG16 backbone and training only a tiny classification head on top. Live webcam demos before and after show the difference.

---
# 1 · Setup & Data

#### Download the dataset from GitHub and split into train / val / test. Images are resized to 224×224 to match what VGG16 expects.

In [ ]:
import os, pathlib, requests
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input

keras.utils.set_random_seed(42)

In [ ]:
API      = 'https://api.github.com/repos/ucla-anderson-SSAI/SSAI/contents/handbags-shoes'
base_dir = pathlib.Path('handbags-shoes')

for category in ('handbags', 'shoes'):
    files = sorted(requests.get(f'{API}/{category}').json(), key=lambda f: f['name'])
    for split, sl in [('train', slice(0,50)), ('validation', slice(50,75)), ('test', slice(75,None))]:
        dst = base_dir / split / category
        os.makedirs(dst, exist_ok=True)
        for f in files[sl]:
            out = dst / f['name']
            if not out.exists():
                out.write_bytes(requests.get(f['download_url']).content)

train_ds = keras.utils.image_dataset_from_directory(base_dir/'train',      image_size=(224,224), batch_size=32, label_mode='binary')
val_ds   = keras.utils.image_dataset_from_directory(base_dir/'validation', image_size=(224,224), batch_size=32, label_mode='binary')
test_ds  = keras.utils.image_dataset_from_directory(base_dir/'test',       image_size=(224,224), batch_size=32, label_mode='binary')

# Quick look at a sample of training images
plt.figure(figsize=(8, 3))
for imgs, labels in train_ds.take(1):
    for i in range(6):
        plt.subplot(1, 6, i+1)
        plt.imshow(imgs[i].numpy().astype('uint8'))
        plt.title('shoe' if labels[i]==1 else 'handbag', fontsize=9)
        plt.axis('off')
plt.tight_layout(); plt.show()

---
# 2 · Webcam Helper

#### Sets up `live_detect()` — a rolling webcam loop that grabs frames, classifies each one, and updates the display in place. We'll reuse this before and after transfer learning.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import time, io, json
from PIL import Image as PILImage

def live_detect(model_fn, title='Live Detection', n_frames=60, interval=0.1):
    """Stream webcam frames into a model and display a live confidence chart.
    model_fn : callable(arr) -> float probability of 'shoe'
    n_frames : how many frames to capture before stopping
    interval : seconds between frames (tune to your GPU/CPU speed)
    """
    js_setup = Javascript('''
        window._stream = null;
        window._video  = null;

        async function startCam() {
            if (window._stream) return 'already running';
            const wrap  = document.createElement('div');
            wrap.style.cssText = 'display:flex;align-items:center;gap:12px;padding:10px;background:#1a1a2e;border-radius:10px;display:inline-flex;';
            const video = document.createElement('video');
            video.style.cssText = 'width:200px;border-radius:6px;';
            const label = document.createElement('div');
            label.id    = 'cam-label';
            label.style.cssText = 'color:#eee;font-family:monospace;font-size:13px;';
            label.textContent   = 'Starting...';
            wrap.appendChild(video);
            wrap.appendChild(label);
            document.body.appendChild(wrap);
            window._video  = video;
            window._stream = await navigator.mediaDevices.getUserMedia({video: true});
            video.srcObject = window._stream;
            await video.play();
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
            return 'started';
        }

        async function grabFrame() {
            const v = window._video;
            const c = document.createElement('canvas');
            c.width = v.videoWidth; c.height = v.videoHeight;
            c.getContext('2d').drawImage(v, 0, 0);
            return c.toDataURL('image/jpeg', 0.7);
        }

        function updateLabel(text) {
            const el = document.getElementById('cam-label');
            if (el) el.innerHTML = text;
        }

        function stopCam() {
            if (window._stream) { window._stream.getTracks().forEach(t => t.stop()); window._stream = null; }
        }
    ''')
    display(js_setup)
    eval_js('startCam()')

    result_display = display('', display_id=True)

    for i in range(n_frames):
        data      = eval_js('grabFrame()')
        img_bytes = b64decode(data.split(',')[1])
        img       = PILImage.open(io.BytesIO(img_bytes)).resize((224, 224))
        arr       = np.expand_dims(np.array(img), axis=0).astype('float32')

        p_shoe = model_fn(arr)
        probs  = {'handbag': 1 - p_shoe, 'shoe': p_shoe}
        top    = max(probs, key=probs.get)
        conf   = probs[top]

        # Update text overlay on the video panel
        label_html = f'<b style="font-size:15px;color:#f1c40f">{top.upper()}</b><br>{conf:.1%} confidence<br><span style="color:#aaa;font-size:11px">frame {i+1}/{n_frames}</span>'
        eval_js(f'updateLabel({json.dumps(label_html)})')

        # Prettier bar chart — no image panel
        fig, ax = plt.subplots(figsize=(5, 2.2))
        fig.patch.set_facecolor('#1a1a2e')
        ax.set_facecolor('#1a1a2e')

        names  = ['handbag', 'shoe']
        vals   = [probs[n] for n in names]
        colors = ['#e74c3c' if n == top else '#2c3e50' for n in names]
        bars   = ax.barh(names, vals, color=colors, height=0.5, edgecolor='none')

        # Confidence labels inside bars
        for bar, v, n in zip(bars, vals, names):
            ax.text(max(v - 0.04, 0.02), bar.get_y() + bar.get_height()/2,
                    f'{v:.1%}', va='center', ha='right', fontsize=11,
                    color='white', fontweight='bold')

        ax.set_xlim(0, 1)
        ax.set_title(f'{title}  →  {top.upper()} ({conf:.1%})',
                     color='white', fontsize=11, pad=8)
        ax.tick_params(colors='#aaa')
        ax.xaxis.set_visible(False)
        for spine in ax.spines.values(): spine.set_visible(False)
        ax.yaxis.set_tick_params(labelcolor='white', labelsize=11)
        plt.tight_layout()

        result_display.update(fig)
        plt.close(fig)
        time.sleep(interval)

    eval_js('stopCam()')
    print('Done.')

---
# 3 · Baseline: CNN Trained from Scratch

#### A small CNN with randomly initialised weights, trained on just 100 images. With so little data and no prior knowledge, it tends to struggle — this sets the bar we want to beat.

In [ ]:
baseline_cnn = models.Sequential([
    layers.Rescaling(1./255, input_shape=(224, 224, 3)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='baseline_cnn')

baseline_cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
baseline_cnn.fit(train_ds, epochs=10, validation_data=val_ds)

acc_baseline = baseline_cnn.evaluate(test_ds)[1]
print(f'Baseline CNN test accuracy: {acc_baseline:.4f}')

---
# 4 · Live Detection — Before Transfer Learning

#### Point your webcam at a handbag or shoe and capture. This is the baseline CNN making a cold guess with no pretrained knowledge.

In [ ]:
def baseline_predict(arr):
    return float(baseline_cnn.predict(arr, verbose=0)[0][0])

try:
    live_detect(baseline_predict, title='Before Transfer Learning (Baseline CNN)')
except Exception as err:
    print(err)

---
# 5 · Transfer Learning

#### We freeze VGG16's ImageNet-trained convolutional base and use it as a fixed feature extractor. Every image is passed through once to produce 7×7×512 feature maps, then we train a small dense head on top of those features. The backbone never changes — only the head learns.

In [ ]:
# Load VGG16 without its classifier top; freeze all weights
vgg_base = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
vgg_base.trainable = False

# Extract features once and cache — makes head training very fast
def extract_features(ds):
    feats, labs = [], []
    for imgs, y in ds:
        feats.append(vgg_base.predict(preprocess_input(imgs), verbose=0))
        labs.append(y.numpy())
    return np.concatenate(feats), np.concatenate(labs)

train_feats, train_labels = extract_features(train_ds)
val_feats,   val_labels   = extract_features(val_ds)
test_feats,  test_labels  = extract_features(test_ds)
print(f'Feature shape: {train_feats.shape}  (samples × 7 × 7 × 512)')

In [ ]:
# Small classification head trained on top of the frozen features
tl_head = models.Sequential([
    layers.Flatten(input_shape=(7, 7, 512)),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
], name='transfer_learning_head')

tl_head.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
tl_head.fit(train_feats, train_labels, epochs=10, batch_size=32,
            validation_data=(val_feats, val_labels))

acc_tl = tl_head.evaluate(test_feats, test_labels)[1]
print(f'Transfer learning test accuracy: {acc_tl:.4f}')

# Quick comparison
plt.bar(['Baseline CNN', 'Transfer Learning'], [acc_baseline, acc_tl], color=['#c0392b', '#2980b9'])
plt.ylabel('Test Accuracy'); plt.ylim(0.4, 1.02)
plt.title('Baseline vs Transfer Learning')
plt.show()

---
# 6 · Live Detection — After Transfer Learning

#### Same webcam demo, now using the transfer learning model. Compare its confidence against what you saw in Section 4.

In [ ]:
def tl_predict(arr):
    feats = vgg_base.predict(preprocess_input(arr.copy()), verbose=0)
    return float(tl_head.predict(feats, verbose=0)[0][0])

try:
    live_detect(tl_predict, title='After Transfer Learning (VGG16 backbone)')
except Exception as err:
    print(err)